In [1]:
import os
import sys
import random
import ast

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from code_mutation.mutation_functions import CodeMutator
from database import MongoDBHelper
from code_inconsistency.code_inconsistency_tester import CodeInconsistencyHumanEvalHelper

In [4]:
# %%script false --no-raise-error

db = MongoDBHelper()
base_qns_db = db.client["Base_Questions_DB"]
question_database = base_qns_db['HumanEval_Open_Ended']

In [5]:
%%script false --no-raise-error
c = 0
f1 = 0
f2 = 0
mutate_failure = {}
f3 = 0
n_f = 0

skippers = set()

for i in range(question_database.count_documents({})):
    task_id = f"HumanEvalo{i}"
    if task_id in skippers:
        continue
    qn = question_database.find_one({"_id": task_id})

    complete_sol = qn['qn'] + "\n" + qn["canon_solution"]
    check = qn['check']
    examples = qn['examples']
    original_qn = qn['qn']
    
    input_metadata = CodeInconsistencyHumanEvalHelper.extract_input_metadata(examples = examples, qn = original_qn)
    variable_metadata = CodeMutator.obtain_variable_types(tree=ast.parse(complete_sol))

    input_metadata = input_metadata | variable_metadata
    example = random.choice(list(examples.keys()))
    func_name = CodeInconsistencyHumanEvalHelper.extract_func_name_from_example(example)    

    if "for" not in complete_sol:
        n_f += 1
        continue

    try:
        namespace = {}
        exec(complete_sol, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])

    except:
        print(f"{task_id} complete solution has issues")
        f1+=1
    
    mutator = CodeMutator()

    try: 
        tree = CodeMutator.parse_through_ast(ast.parse(complete_sol))
        mutated_code = CodeMutator.mutate_for_to_while(ast.parse(tree), input_metadata=input_metadata)
        if CodeMutator.standardize_program(mutated_code) == CodeMutator.standardize_program(complete_sol):
            n_f += 1
            continue
    except Exception as e:
        print(f"{task_id}: Could not mutate the code due to the following error > {type(e),e}")
        mutate_failure[type(e)] = mutate_failure.get(type(e), 0)+1
        f2+=1
        continue    

    try:
        namespace = {}
        exec(mutated_code, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])
        c += 1

    except Exception as e:
        # print(mutated_code)
        print(f"{task_id} mutated solution has issues > {e}")
        print("----")
        print(variable_metadata)
        print(input_metadata)
        print(mutated_code)
        f3 += 1
    except KeyboardInterrupt as e:
        print("###",task_id)
    pass

In [6]:
%%script false --no-raise-error
print(f"{c} test cases were mutated successfully.")
print(f"{f1} test cases failed as complete solution failed.")
print(f"{f2} test cases failed as mutator failed to mutate.")
for key in mutate_failure:
    print(f"    - {key}: {mutate_failure[key]} failures")
print(f"{f3} test cases failed as mutated code failed to pass check.")
print(f"{n_f} test cases contain no for loops for mutation.")
print("All test cases accounted for in mutation tests" if c+f1+f2+f3+n_f == question_database.count_documents({}) else "Some test cases are not accounted for")

## Running singular code inconsistency test

In [7]:
question_database = base_qns_db["HumanEval_Input_Output"]
qn_sample = question_database.find_one({"_id": "HumanEvalTF373"})
examples = qn_sample['examples']
full_sol = qn_sample['full_sol']
input_args = qn_sample['input']['args']
output_args = qn_sample['output']['args']
qn_desc = qn_sample['qn_desc']


for mutation_type in [None, 'for2while']:
    mutated_dict = CodeMutator.mutate_for_code_inconsistency_test(
        mutation_type = mutation_type,
        full_sol = full_sol,
        examples= examples,
        qn_desc= qn_desc,
        input_args= input_args,
        output_args= output_args
    )

    full_sol = mutated_dict['full_sol']
    qn_desc = mutated_dict['qn_desc']
    examples = mutated_dict['examples']

Future exception was never retrieved
future: <Future finished exception=BrokenPipeError(32, 'Broken pipe')>
Traceback (most recent call last):
  File "/Users/jin/.pyenv/versions/3.11.0/lib/python3.11/asyncio/unix_events.py", line 686, in write
    n = os.write(self._fileno, data)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe


check: [0, 2, 2, 2, 5, 5, -5, -5]
MEW [0, 2, 2, 2, 5, 5, -5, -5]
goob [0, 2, 2, 2, 5, 5, -5, -5]
check2: [0, 2, 2, 2, 5, 5, -5, -5]
[0, 2, 2, 2, 5, 5, -5, -5]


IdenticalMutationError: Mutated solution is identical to the original solution.